# Step 2: AAT on top of the locked RAMP baseline (official code + one patch)

AAT = `scripts/ramp_official.sh` method `aat_ramp` (absolute tau) or `aat_rel` (budget-neutral, set `METHOD`) = exactly the `ramp_scratch` flags plus `--aat` (`--aat_relative`)
(`baselines/aat_on_ramp.patch`): per-class adaptive eps (Eq. 4.19) in the Linf and L1 **training** attacks only.
GP fusion, KL loss, lambda=5, lr, 80 epochs and data are identical. Evaluation is the same fixed-eps AutoAttack as the baseline.

**Once:** same secrets and ntfy topic as the baseline notebook (`HF_TOKEN` attached, ntfy app subscribed).

**Each session:** GPU **T4 x2**, Internet **on**, then **Save Version → Save & Run All**. Training pauses at the time budget (run again to resume).
When both seeds finish, the evaluation runs and the result is compared with `baselines/results/ramp_scratch_l5_s{0,1}_eval_autoattack.json`.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
METHOD = 'aat_rel'   # 'aat_rel' = budget-neutral (mean eps == nominal), 'aat_ramp' = absolute tau=0.5
RUN = {'aat_ramp': 'aat_ramp_l5', 'aat_rel': 'aatrel_l5'}[METHOD]   # run-name prefix used by ramp_official.sh
os.environ['TIME_BUDGET_H'] = '10.5'             # training stops cleanly after this and pushes its checkpoint
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

## AAT from scratch (lambda=5, 80 epochs, GP, `--aat`), seeds 0 and 1 in parallel

In [ ]:
def finished(run):
    return os.path.exists(f'external/ramp/trained_models/{run}/log_eval_final.txt')

stage(f'{METHOD}_train', f'bash scripts/ramp_official.sh train2 {METHOD} 0 1')
for s in (0, 1):  # epsilon trajectories: last 3 [aat-eps] lines per run
    f = f'external/ramp/trained_models/{RUN}_s{s}/log_train.txt'
    if os.path.exists(f):
        print(f'--- seed {s}'); print(''.join([l for l in open(f) if l.startswith('[aat-eps]')][-3:]))
if all(finished(f'{RUN}_s{s}') for s in (0, 1)):
    stage(f'{METHOD}_eval', f'bash scripts/ramp_official.sh eval2 {METHOD} 0 1 && '
          f'python scripts/compare_runs.py --names {METHOD} RAMP '
          f'--a \\"runs_official/{RUN}_s*/eval_autoattack.json\\" '
          '--b \\"baselines/results/ramp_scratch_l5_s*_eval_autoattack.json\\"')
else:
    print('Training paused at the time budget. Run the notebook again to resume.')